# StyleMatch all-source corpus, GPU training, and fast index

This notebook is the Colab-only build stage. It fetches every source currently available from the registry, creates one combined parquet file for both literary and rhetorical corpora (no per-chunk text files), optionally fine-tunes mStyleDistance, and builds a cached style + topic profile index. Translation is not used in the default path.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
assert (REPO / 'scripts/multilingual_style_index.py').exists(), REPO
%cd $REPO
print('repo:', Path.cwd())

def run_streamed(cmd, check=True):
    """Colab does not reliably surface child-process stdout in the cell; stream it.
    Reads raw chunks and maps carriage returns to newlines so tqdm progress
    bars (training, encoding) stay visible instead of buffering silently."""
    import os as _os
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    fd = process.stdout.fileno()
    pending = b''
    while True:
        chunk = _os.read(fd, 65536)
        if not chunk:
            break
        pending += chunk
        text = pending.decode('utf-8', errors='replace')
        pending = b''
        print(text.replace('\r\n', '\n').replace('\r', '\n'), end='', flush=True)
    process.wait()
    if check and process.returncode:
        raise RuntimeError(f'command failed with exit {process.returncode}: {" ".join(map(str, cmd))}')
    return process.returncode


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install requests beautifulsoup4 pandas pyarrow sentence-transformers scikit-learn

## 1. Fetch every available source

The first command searches Gutenberg for every registry author in both corpora, including registry-only candidates. The second fetches the curated Chinese, Japanese, French, German, and Russian originals. A missing result is reported and skipped; it is never replaced with a translation.

In [ ]:
# Legacy stylematch_v1 was merged long ago; already-downloaded sources are
# skipped via --skip-covered (Gutenberg) and --skip-existing (multilingual).
run_streamed([sys.executable, 'scripts/fetch_gutendex.py', '--corpus', 'both', '--language', 'en', '--max-works', '0', '--skip-covered'])
run_streamed([sys.executable, 'scripts/fetch_multilingual_sources.py', '--language', 'zh', '--language', 'ja', '--language', 'fr', '--language', 'de', '--language', 'ru', '--language', 'es', '--language', 'it', '--language', 'pl', '--skip-existing'])

In [ ]:
MANIFEST = REPO / 'data/source_registry/source_manifest.csv'
run_streamed([sys.executable, 'scripts/import_source_manifest.py', str(MANIFEST), '--append'])
chunks_path = REPO / 'data/all/meta/all_sources_chunks.parquet'
run_streamed([sys.executable, 'scripts/build_chunk_parquet_from_sources.py', '--corpus', 'both', '--output', str(chunks_path), '--coverage-output', 'data/all/meta/all_sources_coverage.json', '--min-sources', '3', '--min-chunks', '30'])
heldout_path = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
run_streamed([sys.executable, 'scripts/make_source_heldout_splits.py', '--input', str(chunks_path), '--output', str(heldout_path), '--report', 'data/all/meta/all_source_heldout_report.json'])

In [ ]:
import json, pandas as pd
chunks_path = REPO / 'data/all/meta/all_sources_chunks.parquet'
chunks = pd.read_parquet(chunks_path)
print(chunks.shape)
display(chunks.groupby(['language', 'author_or_speaker']).size().sort_values())
coverage = json.loads((REPO / 'data/all/meta/all_sources_coverage.json').read_text())
print('authors:', coverage['n_authors'], 'author-language profiles:', coverage['n_author_language_profiles'], 'sources:', coverage['n_sources'])
display(coverage['not_ready'])
heldout_report = json.loads((REPO / 'data/all/meta/all_source_heldout_report.json').read_text())
print('source-heldout eligible profiles:', heldout_report['eligible_authors'], 'profiles:', heldout_report['n_author_language_profiles'], 'sources:', heldout_report['n_sources'])

## 2. Optional GPU fine-tuning

Run this after the source coverage cell. Every author with at least one source enters the combined profile index. The adapter objective uses same-author, different-source pairs and in-batch negatives; authors with only one source cannot form a valid source-separated positive pair, so they remain in retrieval but are excluded from this specific fine-tuning loss. It never translates or generates source text.

In [ ]:
# Hard preflight: list every registry profile that is trainable, one-source-only, or missing chunks.
run_streamed([sys.executable, 'scripts/finetune_multilingual_style.py', '--input', str(chunks_path), '--audit-only'])
# Train on every author-language profile with at least two sources; the script asserts that none are skipped.
run_streamed([sys.executable, 'scripts/finetune_multilingual_style.py', '--input', str(chunks_path), '--output-dir', 'artifacts/mstyledistance_stylematch_v1', '--pairs-per-author', '500', '--batch-size', '32', '--epochs', '1', '--device', 'cuda', '--language-aware-batches', '--hard-negatives', '--max-seq-length', '256', '--skip-existing'])
# Independent challenger; it is evaluated before any decision to replace or fuse backbones.
run_streamed([sys.executable, 'scripts/finetune_multilingual_style.py', '--input', str(chunks_path), '--model-name', 'Blablablab/multilingual-style-representation', '--output-dir', 'artifacts/multilingual_author_style_v1', '--pairs-per-author', '500', '--batch-size', '16', '--epochs', '1', '--device', 'cuda', '--language-aware-batches', '--hard-negatives', '--max-seq-length', '256', '--skip-existing'])

## 3. Build the cached fast index

The first build encodes corpus chunks on GPU. Re-running it reuses `chunk_embeddings.npz` and `topic_chunk_embeddings.npz`; only new chunk IDs are encoded. The topic model is physically separate from the style model.

In [ ]:
evaluation_runs = {
    'mstyle_pretrained': 'StyleDistance/mstyledistance',
    'mstyle_finetuned': 'artifacts/mstyledistance_stylematch_v1',
    'challenger_pretrained': 'Blablablab/multilingual-style-representation',
    'challenger_finetuned': 'artifacts/multilingual_author_style_v1',
}
for run_name, model_name in evaluation_runs.items():
    run_streamed([sys.executable, 'scripts/style_embedding_recall.py', '--input', str(heldout_path), '--out-dir', f'artifacts/eval_{run_name}', '--model-name', model_name, '--batch-size', '128', '--train-cap', '300', '--eval-splits', 'dev,test', '--device', 'cuda', '--skip-existing'])
run_streamed([sys.executable, 'scripts/style_robust_baseline.py', '--input', str(heldout_path), '--out-dir', 'artifacts/eval_classical', '--skip-existing'])
run_streamed([sys.executable, 'scripts/compare_retrieval_models.py', '--scores', 'mstyle_pretrained=artifacts/eval_mstyle_pretrained/style_embedding_scores.npz:single_centroid_scores', '--scores', 'mstyle_finetuned=artifacts/eval_mstyle_finetuned/style_embedding_scores.npz:single_centroid_scores', '--scores', 'mstyle_prototype=artifacts/eval_mstyle_finetuned/style_embedding_scores.npz:source_prototype_scores', '--scores', 'challenger_pretrained=artifacts/eval_challenger_pretrained/style_embedding_scores.npz:single_centroid_scores', '--scores', 'challenger_finetuned=artifacts/eval_challenger_finetuned/style_embedding_scores.npz:single_centroid_scores', '--scores', 'classical_style=artifacts/eval_classical/style_robust_scores.npz:style_only_fusion', '--output', 'artifacts/model_comparison_v1.json'])
for language in sorted(chunks['language'].unique()):
    n_profiles_language = chunks.loc[chunks['language'].eq(language), 'author_or_speaker'].nunique()
    if n_profiles_language >= 10:
        run_streamed([sys.executable, 'scripts/evaluate_open_set.py', '--input', str(heldout_path), '--model-name', 'artifacts/mstyledistance_stylematch_v1', '--out-dir', f'artifacts/open_set_eval_v1/{language}', '--language', language, '--device', 'cuda'])
run_streamed([sys.executable, 'scripts/evaluate_cross_language.py', '--input', str(heldout_path), '--model-name', 'artifacts/mstyledistance_stylematch_v1', '--out-dir', 'artifacts/cross_language_eval_v1', '--device', 'cuda'])
decade_validation = None
if 'decade' in chunks.columns and chunks['decade'].fillna('').astype(str).ne('').any():
    decade_rc = run_streamed([sys.executable, 'scripts/evaluate_decades.py', '--input', str(chunks_path), '--model-name', 'artifacts/mstyledistance_stylematch_v1', '--out-dir', 'artifacts/decade_eval_v1', '--device', 'cuda'], check=False)
    if decade_rc == 0:
        decade_validation = 'artifacts/decade_eval_v1/decade_metrics.json'
build_index = [sys.executable, 'scripts/multilingual_style_index.py', 'build', '--input', str(chunks_path), '--out-dir', 'artifacts/multilingual_style_index_v1', '--model-name', 'artifacts/mstyledistance_stylematch_v1', '--topic-model-name', 'intfloat/multilingual-e5-base', '--batch-size', '128', '--per-source-cap', '50', '--profile-cap', '600', '--profile-strategy', 'single_centroid', '--artifact-version', 'baseline_v1', '--heldout-report', 'data/all/meta/all_source_heldout_report.json', '--open-set-calibration-dir', 'artifacts/open_set_eval_v1', '--model-label', 'mstyle_finetuned', '--model-comparison', 'artifacts/model_comparison_v1.json', '--device', 'cuda']
if decade_validation:
    build_index.extend(['--decade-validation', decade_validation])
run_streamed(build_index)

# Leave-one-source-out rotation over the exact chunks retained by the index cache.
# The script aligns cache and parquet by chunk_id and performs matrix math only.
run_streamed([sys.executable, 'scripts/evaluate_loso_retrieval.py', '--input', str(chunks_path), '--model-name', 'artifacts/mstyledistance_stylematch_v1', '--embedding-cache', 'artifacts/multilingual_style_index_v1/chunk_embeddings.npz', '--output', 'artifacts/loso_eval_v1/loso_metrics.json'])


In [ ]:
index_metadata = json.loads((REPO / 'artifacts/multilingual_style_index_v1/metadata.json').read_text())
assert len(index_metadata['languages']) > 1, f"Cross-language Part 3 requires multilingual chunks; found {index_metadata['languages']}"
query_text = 'The institution changed slowly, while ordinary people learned to live with its contradictions.'
run_streamed([sys.executable, 'scripts/multilingual_style_index.py', 'benchmark', '--index-dir', 'artifacts/multilingual_style_index_v1', '--language', 'en', '--mode', 'within', '--text', query_text, '--runs', '30', '--output', 'artifacts/multilingual_style_index_v1/latency_gpu.json', '--device', 'cuda'])
run_streamed([sys.executable, 'scripts/multilingual_style_index.py', 'benchmark', '--index-dir', 'artifacts/multilingual_style_index_v1', '--language', 'en', '--mode', 'within', '--text', query_text, '--runs', '30', '--output', 'artifacts/multilingual_style_index_v1/latency_cpu.json', '--device', 'cpu'])
run_streamed([sys.executable, 'scripts/multilingual_style_index.py', 'query', '--index-dir', 'artifacts/multilingual_style_index_v1', '--language', 'en', '--mode', 'within', '--text', query_text, '--top-k', '3', '--device', 'cuda'])
# Cross-language results remain separate by target language until ordered language-pair calibration exists.
run_streamed([sys.executable, 'scripts/multilingual_style_index.py', 'query', '--index-dir', 'artifacts/multilingual_style_index_v1', '--language', 'en', '--mode', 'cross', '--text', query_text, '--top-k', '3', '--device', 'cuda'])

In [ ]:
# Metadata and split audits produce explicit reports; missing optional group coverage is reported, not hidden.
run_streamed([sys.executable, 'scripts/audit_source_metadata.py', '--corpus', 'both', '--output', 'artifacts/baseline_v1/source_metadata_audit.json'])
for group_column in ['topic', 'domain', 'register', 'decade']:
    if group_column in chunks and chunks[group_column].fillna('').astype(str).nunique() >= 3:
        run_streamed([sys.executable, 'scripts/make_group_heldout_splits.py', '--input', str(chunks_path), '--group-column', group_column, '--output', f'data/all/meta/{group_column}_heldout_splits.parquet', '--report', f'data/all/meta/{group_column}_heldout_report.json'], check=False)
run_streamed([sys.executable, 'scripts/evaluate_language_id.py', '--input', 'data/eval/language_id_cases.csv', '--output', 'artifacts/baseline_v1/language_id_report.json'])
open_set_metric_paths = sorted(Path('artifacts/open_set_eval_v1').glob('*/open_set_metrics.json'))
if not open_set_metric_paths:
    raise RuntimeError('No open-set metrics were produced')
open_set_cli = [item for path in open_set_metric_paths for item in ('--open-set-metrics', str(path))]
extra_open_set_cli = [item for path in open_set_metric_paths for item in ('--extra-artifact', str(path))]
run_streamed([sys.executable, 'scripts/audit_release_readiness.py', '--chunks', str(chunks_path), '--heldout-report', 'data/all/meta/all_source_heldout_report.json', '--training-config', 'artifacts/mstyledistance_stylematch_v1/training_config.json', '--index-metadata', 'artifacts/multilingual_style_index_v1/metadata.json', '--embedding-metrics', 'artifacts/eval_mstyle_finetuned/style_embedding_metrics.json', '--extra-artifact', 'artifacts/model_comparison_v1.json', '--extra-artifact', 'artifacts/cross_language_eval_v1/cross_language_metrics.json', '--extra-artifact', 'artifacts/multilingual_style_index_v1/latency_cpu.json', *extra_open_set_cli, '--output-dir', 'artifacts/baseline_v1', '--version', 'baseline_v1', '--strict'])
run_streamed([sys.executable, 'scripts/check_release_gates.py', '--retrieval-metrics', 'artifacts/eval_mstyle_finetuned/style_embedding_metrics.json', *open_set_cli, '--latency', 'artifacts/multilingual_style_index_v1/latency_cpu.json', '--heldout-report', 'data/all/meta/all_source_heldout_report.json', '--index-metadata', 'artifacts/multilingual_style_index_v1/metadata.json', '--source-metadata-audit', 'artifacts/baseline_v1/source_metadata_audit.json', '--language-id-report', 'artifacts/baseline_v1/language_id_report.json', '--readiness-report', 'artifacts/baseline_v1/readiness_report.json', '--output', 'artifacts/baseline_v1/release_gates.json'])

## 4. Artifact check

Keep the parquet, model directory, and index directory in Drive. Do not save chunk-level `.txt` files. The index directory is what the future web app loads.

In [ ]:
!du -sh data/all/meta/all_sources_chunks.parquet artifacts/mstyledistance_stylematch_v1 artifacts/multilingual_author_style_v1 artifacts/multilingual_style_index_v1 artifacts/baseline_v1
print('training/index artifacts are under:', REPO / 'artifacts')
print('model decision:', json.loads((REPO / 'artifacts/model_comparison_v1.json').read_text())['decision'])
print('private beta ready:', json.loads((REPO / 'artifacts/baseline_v1/release_gates.json').read_text())['private_beta_ready'])